In [0]:
_checkpoints = "dbfs:/Volumes/workspace/retails/raw_data/_checkpoints/retails/dev/gold/categories"

In [0]:
# -- # Read clean categories from silver
silver_df = (
    spark.readStream \
        .format("delta") \
        .option("readChangeFeed", "true") \
        .table("retails.silver.categories_cleaned")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, sha2, concat_ws

# Prepare dataframe for gold SCD2
silver_df = (
    silver_df
        .select(
            "category_id",
            "category_department_id",
            "category_name",
            "op",
            "_change_type"
        ) \
            
        .filter(
            col("_change_type").isin(
                "insert",
                "update_postimage",
                "delete"
            )
        ) \

        .withColumn("is_active", lit(True).cast("boolean")) \
        .withColumn("effective_from", current_timestamp()) \
        .withColumn(
            "effective_to",
            lit(None).cast("timestamp")
        ) \
        .withColumn("is_current", lit(True).cast("boolean")) \
        .withColumn("created_ts", current_timestamp()) \
        .withColumn("updated_ts", current_timestamp()) \
        .withColumn("record_hash",
                        sha2(
                            concat_ws(
                                "||",
                                col("category_id"),
                                col("category_department_id"),
                                col("category_name")
                            ),
                            256
                        )
                    ))
    


In [0]:
# STEP-1: Expire old current rows
MERGE_EXPIRE = """
MERGE INTO retails.gold.dim_categories t
USING silver_categories_vw s

ON t.category_id = s.category_id
AND t.is_current = true

WHEN MATCHED AND s.op = 'UPDATE'
THEN UPDATE SET
    t.is_active = false,
    t.is_current = false,
    t.effective_to = current_timestamp(),
    t.updated_ts = current_timestamp()

WHEN MATCHED AND s.op = 'DELETE'
THEN UPDATE SET 
    t.is_active = false,
    t.is_current = false,
    t.effective_to = current_timestamp(),
    t.updated_ts = current_timestamp()
"""

In [0]:
# STEP-2: Insert new versions
INSERT_NEW = """
INSERT INTO retails.gold.dim_categories(
    category_id,
    category_department_id,
    category_name,
    is_active,
    effective_from,
    effective_to,
    is_current,
    created_ts,
    updated_ts)

SELECT
    s.category_id,
    s.category_department_id,
    s.category_name,
    true,
    current_timestamp(),
    CAST(NULL AS TIMESTAMP),
    true,
    current_timestamp(),
    current_timestamp()

FROM silver_categories_vw s

WHERE s.op IN ('INSERT', 'UPDATE')
"""

In [0]:
# foreachBatch function
def upsert_to_gold(batch_df, batch_id):

    batch_df.createOrReplaceTempView(
        "silver_categories_vw"
    )

    spark.sql(MERGE_EXPIRE)

    spark.sql(INSERT_NEW)


In [0]:
# Start streaming query
query = (
    silver_df.writeStream
        .foreachBatch(upsert_to_gold)
        .option(
            "checkpointLocation",
            _checkpoints
        )
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()

In [0]:
# dbutils.fs.ls(_checkpoints)
# dbutils.fs.rm(_checkpoints, True)

In [0]:
%sql
-- make silver cleaned table CDF enabled

-- ALTER TABLE retails.silver.categories_cleaned
-- SET TBLPROPERTIES (
--     delta.enableChangeDataFeed = true
-- )

-- DESCRIBE TABLE EXTENDED retails.silver.categories_cleaned;

In [0]:
%sql
-- select * from retails.gold.dim_categories;